# ***LCO2 Transport Simulation***

### Sources :
1. The shipping logistics are based on - [IEAGHG PH4-30 Ship Transport Report] - https://publications.ieaghg.org/docs/General_Docs/Reports/PH4-30%20Ship%20Transport.pdf
2.  The Ship parameters are based on - Northern pioneer class for S10, MV Active/Amdeus for S30, MOL–Mitsubishi  LCO2 Carrier S50

## **SETTING UP**

In [ ]:
import os
import sys
import time
import datetime
import warnings

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings('ignore')
np.random.seed(2025)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **SHIP PARAMTERS**

In [ ]:
CO2_DENSITY_T_M3       = 1.155
CO2_LATENT_HEAT_KJ     = 340.0
CO2_CARGO_TEMP_C       = -50.0
CO2_CARGO_PRESSURE_BAR = 6.83
BOR_DESIGN_PCT_DAY     = 0.20

SOURCES : Wartsila Encyclopedia of Marine & Energy Technology, MAN Energy Solutions, IIMO MEPC 60/4/33; IMO MEPC.308(73)


In [ ]:
SHIPS = {
    'S10': {
        'size_class':        'S10',
        'label':             '10,000 t CO2 carrier',
        'cargo_capacity_t':  8662.5,
        'cargo_capacity_m3': round(8662.5 / CO2_DENSITY_T_M3),
        'Lpp': 130.0, 'B': 21.5, 'T_laden': 7.03, 'T_ballast': 4.80,
        'Cb': 0.720, 'Cp': 0.745, 'Cm': 0.968,
        'disp_laden_t': 14500, 'disp_ballast_t': 9900,
        'Cadm': 425, 'MCR_kW': 6000, 'SFOC_ref_g_kWh': 185,
        'D_prop_m': 4.8, 'n_prop_rps': 2.58, 'w_wake': 0.23, 't_thrust': 0.19,
        'eta_R': 1.00, 'eta_S': 0.985,
        'S_tank_m2': 3_088,
        'U_overall_W_m2K': 0.15,
        'loading_rate_t_hr': 900, 'unloading_rate_t_hr': 700,
    },
    'S30': {
        'size_class':        'S30',
        'label':             '30,000 t CO2 carrier',
        'cargo_capacity_t':  25410,
        'cargo_capacity_m3': round(25410 / CO2_DENSITY_T_M3),
        'Lpp': 185.0, 'B': 30.5, 'T_laden': 9.84, 'T_ballast': 6.04,
        'Cb': 0.738, 'Cp': 0.761, 'Cm': 0.970,
        'disp_laden_t': 42_000, 'disp_ballast_t': 25_800,
        'Cadm': 445, 'MCR_kW': 11500, 'SFOC_ref_g_kWh': 181,
        'D_prop_m': 6.4, 'n_prop_rps': 2.12, 'w_wake': 0.26, 't_thrust': 0.21,
        'eta_R': 1.02, 'eta_S': 0.986,
        'S_tank_m2': 6_423,
        'U_overall_W_m2K': 0.13,
        'loading_rate_t_hr': 1_500, 'unloading_rate_t_hr': 1_200,
    },
    'S50': {
        'size_class':        'S50',
        'label':             '50,000 t CO2 carrier',
        'cargo_capacity_t':  57750,
        'cargo_capacity_m3': round(50_000 / CO2_DENSITY_T_M3),
        'Lpp': 230.0, 'B': 37.0, 'T_laden': 13.5, 'T_ballast': 7.8,
        'Cb': 0.748, 'Cp': 0.771, 'Cm': 0.970,
        'disp_laden_t': 72_000, 'disp_ballast_t': 47_500,
        'Cadm': 462, 'MCR_kW': 16000, 'SFOC_ref_g_kWh': 177,
        'D_prop_m': 7.8, 'n_prop_rps': 1.75, 'w_wake': 0.28, 't_thrust': 0.22,
        'eta_R': 1.02, 'eta_S': 0.987,
        'S_tank_m2': 9_028,
        'U_overall_W_m2K': 0.12,
        'loading_rate_t_hr': 2_200, 'unloading_rate_t_hr': 1_800,
    },
}
ship_speed = 15.0 #knots

## **ROUTES**

In [ ]:
# Importing waypoints extracted from AIS data
import json
import os

WAYPOINTS_JSON_PATH = '/content/drive/MyDrive/LCO2 Transport Project Final/route_waypoints.json'

def load_ais_routes(json_path):
    with open(json_path, 'r') as f:
        raw_routes = json.load(f)
    routes_dict = {}
    for route_id, data in raw_routes.items():
        wps = [(float(pt[0]), float(pt[1])) for pt in data['waypoints']]
        routes_dict[route_id] = {
            'name': data.get('name', route_id),
            'origin': data.get('origin', 'AIS Extracted Origin'),
            'destination': data.get('destination', 'AIS Extracted Destination'),
            'waypoints': wps
        }

    return routes_dict

ROUTES = load_ais_routes(WAYPOINTS_JSON_PATH)

In [ ]:
routes_to_remove = {'GOM-HSC_EXM','GOM-PAB_EXM','GOM-PLQ_EXM'}
for r_id in routes_to_remove:
  ROUTES.pop(r_id,None)

In [ ]:
# Looking at the routes
for route_id, data in ROUTES.items():
    print(f" Route ID:    {route_id}")
    print(f" Name:        {data['name']}")
    print(f" Origin:      {data['origin']}")
    print(f" Destination: {data['destination']}")

    wps = data['waypoints']
    print(f" Total Waypoints: {len(wps)}")
    for wp in wps :
      print(wp)
    print('\n')

 Route ID:    GOM-HSC_BB
 Name:        GOM-HSC_BB
 Origin:      [29.35, -94.77]
 Destination: [29.6, -94.0]
 Total Waypoints: 8
(29.35, -94.77)
(29.3078, -94.6278)
(29.2327, -94.4927)
(29.165, -94.3539)
(29.2605, -94.2295)
(29.3553, -94.1047)
(29.4633, -93.9867)
(29.6, -94.0)


 Route ID:    GOM-PAB_BB
 Name:        GOM-PAB_BB
 Origin:      [29.893, -93.938]
 Destination: [29.6, -94.0]
 Total Waypoints: 8
(29.893, -93.938)
(29.804, -93.955)
(29.7359, -93.8759)
(29.6479, -93.825)
(29.5634, -93.7634)
(29.505, -93.8299)
(29.5211, -93.9311)
(29.6, -94.0)


 Route ID:    GOM-PLQ_BB
 Name:        GOM-PLQ_BB
 Origin:      [30.289, -91.221]
 Destination: [29.6, -94.0]
 Total Waypoints: 18
(30.289, -91.221)
(30.115, -91.0005)
(30.0504, -90.5604)
(29.92, -90.13)
(29.7422, -90.0152)
(29.4149, -89.605)
(29.1082, -89.269)
(28.7993, -89.2633)
(28.765, -89.3248)
(28.795, -89.8218)
(28.765, -90.3768)
(28.725, -90.9884)
(28.715, -91.6154)
(28.715, -92.2475)
(28.715, -92.8797)
(28.8495, -93.4195)
(29.32

## **ECONOMIC PARAMETERS**

### **SOURCES :**
Drewry, Ship Operating Costs Annual Review
2022/23; Roussanaly et al. (2021), Energies
14(18):5635, Ship & Bunker (G20-VLSFO, Houston VLSFO
price indices), IMO MEPC.308(73) (2018); IMO MEPC.364(79)
(2022)

In [ ]:
# Operating Costs Paramters

ECON = {
    'ship_opex_usd_per_day': {'S10': 5500, 'S30': 8000, 'S50': 10500},
    'fuel_price_usd_per_t': 535.0,
    'carbon_price_usd_per_t_co2': 100.0,
    'port_cost_per_call_usd': {'S10': 18650, 'S30': 22180, 'S50': 26530},
    'buffer_storage_opex_usd_per_t_cap_yr': 29.0,
    'buffer_storage_days': 5.0,   # days of throughput held in the buffer tank
    'liquefaction_opex_usd_per_t': 22.0,   #Liquefaction OPEX
    'co2_per_t_vlsfo': 3.151,  # t CO2/t VLSFO [IMO MEPC.282(70), 0.5% S fuel] - Emission factor

}


## **VOYAGE SIMULATION MATH AND PHYSICS**

In [ ]:
def haversine_nm(lat1, lon1, lat2, lon2):           # Great-circle distance in nautical miles
    R = 3440.065
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def initial_bearing(lat1, lon1, lat2, lon2):       # Initial bearing (degrees true) from point 1 -> point 2
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dlam = np.radians(lon2 - lon1)
    x = np.sin(dlam) * np.cos(phi2)
    y = np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(dlam)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


class RouteInterpolator:        # Interpolates (lat, lon, heading) from cumulative distance sailed along a polyline of waypoints
    def __init__(self, waypoints):
        self.wps = waypoints
        self.cum_dists = [0.0]
        self.bearings = []
        for i in range(len(waypoints) - 1):
            d = haversine_nm(*waypoints[i], *waypoints[i + 1])
            self.cum_dists.append(self.cum_dists[-1] + d)
            self.bearings.append(initial_bearing(*waypoints[i], *waypoints[i + 1]))
        self.total_dist_nm = self.cum_dists[-1]

    def state_at(self, dist_nm):
        dist_nm = float(np.clip(dist_nm, 0.0, self.total_dist_nm))
        seg = 0
        for i in range(len(self.wps) - 1):
            if dist_nm <= self.cum_dists[i + 1]:
                seg = i
                break
        else:
            seg = len(self.wps) - 2
        seg_start = self.cum_dists[seg]
        seg_len = self.cum_dists[seg + 1] - seg_start
        frac = (dist_nm - seg_start) / seg_len if seg_len > 1e-6 else 1.0
        lat1, lon1 = self.wps[seg]
        lat2, lon2 = self.wps[seg + 1]
        lat = lat1 + frac * (lat2 - lat1)
        lon = lon1 + frac * (lon2 - lon1)
        return float(lat), float(lon), float(self.bearings[seg])

### **SOURCES :**
- ITTC - Calculation of the Weather Factor for Decrease of Ship Speed in Wind and Waves (2024)
- Faltinsen et al. (1980) - Prediction of Resistance and Propulsion of a Ship in a Seaway:
- Pierson & Moskowitz (1964) - Fully developed wind-sea relationship:
- Moskowitz (1964) - Fully developed sea spectra:  https://doi.org/10.1029/JZ069i024p05181
- LCO₂ boil-off/re-liquefaction for transport ships (2017): 10.1016/j.ijggc.2017.10.008


In [ ]:
# Physics Models

BEAUFORT_THRESHOLDS = [0.3, 1.5, 3.4, 5.4, 7.9, 10.7, 13.8, 17.1, 20.7, 24.4, 28.4, 32.6]

MONTH_TO_SEASON = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn',
} # delay near port with season

HURRICANE_OPS_FACTOR = {'Winter': 1.0, 'Spring': 1.0, 'Summer': 0.932, 'Autumn': 0.975}
SEASONAL_PORT_WAIT_HR = {'Winter': 2.5, 'Spring': 1.2, 'Summer': 0.8, 'Autumn': 1.5}

def seawater_props(SST_C, salinity_ppt=35.0):  # seawater density and kinematic viscosity
    SST_C = float(np.clip(SST_C, 0.0, 35.0))
    rho_w = 1025.0 - 0.180 * (SST_C - 15.0) + 0.0008 * salinity_ppt
    nu = 1.83e-6 / (1.0 + 0.0337 * SST_C + 0.000221 * SST_C**2)
    nu = float(np.clip(nu, 0.55e-6, 1.95e-6))
    rho_w = float(np.clip(rho_w, 1018.0, 1030.0))
    return rho_w, nu


def ittc_frictional_resistance(V_kn, ship, draft_m, SST_C):
    Lpp = ship['Lpp']; B = ship['B']; Cb = ship['Cb']
    rho_w, nu = seawater_props(SST_C)
    V_ms = V_kn * 0.5144
    if V_ms < 0.01:
        return 0.0, 0.0, 0.0
    S = Lpp * (2 * draft_m + B) * np.sqrt(Cb) * 0.93
    Re = V_ms * Lpp / nu
    C_F = 0.075 / (np.log10(Re) - 2.0)**2
    R_F = 0.5 * rho_w * S * V_ms**2 * C_F / 1000.0
    return float(R_F), float(S), float(C_F)


def admiralty_power(V_kn, ship, condition='laden', draft_m=None, SST_C=15.0):   # Calm water power via Admiralty coefficient + ITTC SST correction.
    disp = ship['disp_laden_t'] if condition == 'laden' else ship['disp_ballast_t']
    if draft_m is None:
        draft_m = ship['T_laden'] if condition == 'laden' else ship['T_ballast']
    P_kW = (disp**(2/3)) * (V_kn**3) / ship['Cadm']
    _, _, CF_sst = ittc_frictional_resistance(V_kn, ship, draft_m, SST_C)
    _, _, CF_ref = ittc_frictional_resistance(V_kn, ship, draft_m, 15.0)
    sst_factor = CF_sst / CF_ref if CF_ref > 1e-9 else 1.0
    P_kW_corrected = P_kW * (0.25 + 0.75 * sst_factor)
    return float(P_kW_corrected)


def wageningen_eta(V_kn, ship, P_kW):
    w = ship['w_wake']; t = ship['t_thrust']
    eta_R = ship['eta_R']; eta_S = ship['eta_S']
    D = ship['D_prop_m']; n = ship['n_prop_rps']
    V_ms = V_kn * 0.5144
    V_A = V_ms * (1 - w)
    J = V_A / (n * D) if n > 0 else 0.6
    J = float(np.clip(J, 0.3, 1.0))
    eta_O = float(np.clip(0.68 - 0.15 * (J - 0.70)**2, 0.40, 0.72))
    eta_H = (1 - t) / (1 - w) if (1 - w) > 0 else 1.0
    eta_D = eta_H * eta_O * eta_R
    eta_total = eta_D * eta_S
    eta_total = float(np.clip(eta_total, 0.40, 0.78))
    return eta_total, eta_D, eta_O, eta_H


def engine_sfoc(P_kW, ship):  # Fuel efficiency
    MCR = ship['MCR_kW']
    SFOC_ref = ship['SFOC_ref_g_kWh']
    eta = float(np.clip(P_kW / MCR, 0.15, 1.05))
    if eta < 0.25:
        sfoc_factor = 1.55
    else:
        sfoc_factor = 0.455 / eta**2 - 0.710 / eta + 1.280
    SFOC = float(np.clip(SFOC_ref * sfoc_factor, 148, 295))
    return SFOC, eta


def kwon_speed_loss(V_design, Hs, Tp, wind_ms, mu_deg, ship):
    Cb = ship['Cb']; Lpp = ship['Lpp']
    Bf = next((i for i, t in enumerate(BEAUFORT_THRESHOLDS) if wind_ms < t), 12)
    alpha = (0.7 + 1.3 * Cb if Cb <= 0.65
             else -4.5 * Cb + 4.0 if Cb <= 0.75
             else 0.054)
    Cu_map = {0: 0, 1: 0, 2: 0, 3: 0.5, 4: 1.0, 5: 1.5, 6: 2.5,
              7: 4.0, 8: 5.5, 9: 7.0, 10: 9.0, 11: 11.0, 12: 14.0}
    Cu = Cu_map.get(min(Bf, 12), 14.0)
    mu = mu_deg
    if mu <= 30: f_h = 1.00 + 0.35 * (mu / 30)
    elif mu <= 60: f_h = 1.35 - 0.15 * ((mu - 30) / 30)
    elif mu <= 90: f_h = 1.20 - 0.40 * ((mu - 60) / 30)
    elif mu <= 120: f_h = 0.80 - 0.30 * ((mu - 90) / 30)
    elif mu <= 150: f_h = 0.50 - 0.35 * ((mu - 120) / 30)
    else: f_h = 0.15 + 0.10 * ((180 - mu) / 30)
    steep = 1.0 + 0.6 * (Hs / np.sqrt(Lpp))
    dV_pct = float(np.clip(alpha * Cu * f_h * steep, 0.0, 45.0))
    V_actual = float(max(V_design * (1 - dV_pct / 100), V_design * 0.40))
    return V_actual, dV_pct


def faltinsen_wave_resistance_kN(Hs, V_kn, ship, SST_C=28.0):   # "Added wave resistance [Faltinsen et al. 1980]
    rho_w, _ = seawater_props(SST_C)
    g = 9.81
    A_s = Hs / 2.0
    B = ship['B']; Lpp = ship['Lpp']; Cb = ship['Cb']
    V_ms = V_kn * 0.5144
    Fn = V_ms / np.sqrt(g * Lpp)
    C_r = 0.75 + 0.18 * Cb
    Fn_corr = 1.0 + 3.5 * Fn
    R_aw = (C_r * rho_w * g * A_s**2 * B**2 / Lpp) * Fn_corr
    return float(np.clip(R_aw / 1000, 0, 1500))


def boil_off_rate(ship, SST_C, Hs, dt_hr=1.0):
    U = ship['U_overall_W_m2K']
    A_tank = ship['S_tank_m2']
    T_cargo = CO2_CARGO_TEMP_C
    h_fg = CO2_LATENT_HEAT_KJ * 1000
    delta_T = SST_C - T_cargo
    f_wave = 1.0 + 0.04 * Hs
    Q_in_W = U * A_tank * delta_T * f_wave
    m_bog_kg_s = Q_in_W / h_fg
    m_bog_kg = m_bog_kg_s * dt_hr * 3600
    return float(np.clip(m_bog_kg, 0, 1e6))


def wind_speed_from_wind_wave_height(Hs_ww): #Pierson-Moskowitz fully-developed-sea relation : Hs = (0.21*U^2/g)^-1
    g = 9.81
    U = np.sqrt(np.clip(Hs_ww, 0, None) * g / 0.21)
    return float(np.clip(U, 0.0, 35.0))

## **LOADING COPERNICUS WEATHER DATA**

In [ ]:
import glob

WEATHER_DIR = '/content/drive/MyDrive/LCO2 Transport Project Final'
OUTPUT_DIR = os.path.join(WEATHER_DIR, 'simulation_files')
os.makedirs(OUTPUT_DIR, exist_ok=True)

WEATHER_FILE_PATTERN = 'gulf_merged_2025_*.nc'
WEATHER_VARS = ['uo', 'vo', 'thetao', 'swh', 'mwd', 'wind_wave_height']


def _routes_bbox(buffer_deg=0.35): # bound the space to the coordinates required (smaller box)
    lats, lons = [], []
    for r in ROUTES.values():
        for lat, lon in r['waypoints']:
            lats.append(lat); lons.append(lon)
    return (min(lons) - buffer_deg, max(lons) + buffer_deg,
            min(lats) - buffer_deg, max(lats) + buffer_deg)


def get_downloaded_weather_files(weather_dir):
    pattern = os.path.join(weather_dir, WEATHER_FILE_PATTERN)
    files = sorted(glob.glob(pattern))
    months = sorted(int(os.path.basename(f)[-5:-3]) for f in files)
    return files


class CopernicusWeather:

    def __init__(self, weather_dir=None, fill_land_with_nearest_ocean=True,
                 max_time_gap_hr=3.0):
        import xarray as xr
        weather_dir = WEATHER_DIR
        files = get_downloaded_weather_files(weather_dir)
        lon_min, lon_max, lat_min, lat_max = _routes_bbox()
        print(f'  [Weather] cropping to route box lon {lon_min:.2f}..{lon_max:.2f}, '
              f'lat {lat_min:.2f}..{lat_max:.2f} before loading')

        chunks = {v: [] for v in WEATHER_VARS}     # per-variable list of monthly (time, lat, lon) arrays
        times, lat_ref, lon_ref = [], None, None
        for f in files:
            try:
                ds = xr.open_dataset(f)
            except Exception as e:
                raise RuntimeError(f'Could not open {f}: {e}\n'
                                   'Either the file is incomplete/corrupt, or the '
                                   'NetCDF reader is missing ') from e
            ds = ds[WEATHER_VARS]
            if 'depth' in ds.dims:
                ds = ds.isel(depth=0)
            ds = ds.drop_vars('depth', errors='ignore')

            lat_name = 'latitude' if 'latitude' in ds.coords else 'lat'
            lon_name = 'longitude' if 'longitude' in ds.coords else 'lon'
            lat_vals, lon_vals = ds[lat_name].values, ds[lon_name].values
            lat_sl = slice(lat_min, lat_max) if lat_vals[0] <= lat_vals[-1] else slice(lat_max, lat_min)
            lon_sl = slice(lon_min, lon_max) if lon_vals[0] <= lon_vals[-1] else slice(lon_max, lon_min)
            ds = ds.sel({lat_name: lat_sl, lon_name: lon_sl})
            if ds.sizes[lat_name] == 0 or ds.sizes[lon_name] == 0:
                raise RuntimeError(
                    f'{os.path.basename(f)}: route box lies outside the downloaded area '
                    f'(file covers lat {lat_vals.min():.1f}..{lat_vals.max():.1f}, '
                    f'lon {lon_vals.min():.1f}..{lon_vals.max():.1f}).')
            ds = ds.load()       # only the cropped box is loaded from Drive

            if lat_ref is None:
                lat_ref, lon_ref = ds[lat_name].values.astype(float), ds[lon_name].values.astype(float)
            elif (ds[lat_name].size != lat_ref.size or ds[lon_name].size != lon_ref.size or
                  not np.allclose(ds[lat_name].values, lat_ref) or not np.allclose(ds[lon_name].values, lon_ref)):
                raise RuntimeError(f'{os.path.basename(f)} is on a different lat/lon grid than the first file.')

            times.append(ds['time'].values)
            for v in WEATHER_VARS:
                chunks[v].append(ds[v].transpose('time', lat_name, lon_name).values)
            print(f'    loaded {os.path.basename(f)}  ({ds.sizes["time"]} steps)')
            ds.close()
            del ds

        t_all = np.concatenate(times).astype('datetime64[ns]')
        t_sorted, keep = np.unique(t_all, return_index=True)        # sort + drop duplicated timestamps
        reorder = not np.array_equal(keep, np.arange(t_all.size))

        self._lat_name, self._lon_name = lat_name, lon_name
        self._lat, self._lon = lat_ref, lon_ref
        self._v = {}
        for v in WEATHER_VARS:                                       # one variable at a time -> low peak RAM
            arr = np.concatenate(chunks.pop(v), axis=0)
            self._v[v] = arr[keep] if reorder else arr
            del arr
        self._t_ns = t_sorted.astype('int64')
        self._dlat = float(np.median(np.abs(np.diff(self._lat)))) if self._lat.size > 1 else 0.083
        self._dlon = float(np.median(np.abs(np.diff(self._lon)))) if self._lon.size > 1 else 0.083
        self._max_gap_ns = int(max_time_gap_hr * 3600e9)
        self._fill = bool(fill_land_with_nearest_ocean)

        # cells that have data for every variable at some time = "ocean"
        valid = np.ones((self._lat.size, self._lon.size), dtype=bool)
        for v in WEATHER_VARS:
            valid &= np.isfinite(self._v[v]).any(axis=0)
        if not valid.any():
            raise RuntimeError('No cell in the route box has weather data -- check the route box / files.')
        self._valid = valid

        # for every cell, the indices of the nearest "ocean" cell
        from scipy.ndimage import distance_transform_edt
        _, (self._nn_i, self._nn_j) = distance_transform_edt(~valid, return_indices=True)

        mb = sum(a.nbytes for a in self._v.values()) / 1e6
        print(f'  [Weather] ready: {self._t_ns.size} time steps '
              f'({pd.Timestamp(self._t_ns[0])} .. {pd.Timestamp(self._t_ns[-1])}), '
              f'grid {self._lat.size} lat x {self._lon.size} lon, '
              f'{int((~valid).sum())} no-data (land/river) cells of {valid.size}, ~{mb:.0f} MB in RAM')

    def _time_index(self, timestamp):
        t = pd.Timestamp(timestamp).value                       # ns since epoch
        k = int(np.searchsorted(self._t_ns, t))
        if k <= 0:
            k = 0
        elif k >= self._t_ns.size:
            k = self._t_ns.size - 1
        elif abs(self._t_ns[k - 1] - t) <= abs(self._t_ns[k] - t):
            k -= 1
        if abs(int(self._t_ns[k]) - t) > self._max_gap_ns:
            raise RuntimeError(f'{pd.Timestamp(timestamp)} is outside the downloaded weather period '
                               f'({pd.Timestamp(self._t_ns[0])} .. {pd.Timestamp(self._t_ns[-1])}).')
        return k

    def at(self, dist_nm, heading_deg, lat, lon, timestamp):
        k = self._time_index(timestamp)
        i = int(np.abs(self._lat - lat).argmin())
        j = int(np.abs(self._lon - lon).argmin())

        on_native_cell = (abs(self._lat[i] - lat) <= 0.6 * self._dlat and
                          abs(self._lon[j] - lon) <= 0.6 * self._dlon)
        source = 'copernicus_exact' if on_native_cell else 'copernicus_nearest'

        if not self._valid[i, j]:                               # land / river cell
            if not self._fill:
                raise RuntimeError(f'No ocean data at lat {lat:.3f}, lon {lon:.3f} (land/river cell) '
                                   'and fill_land_with_nearest_ocean=False.')
            i, j = int(self._nn_i[i, j]), int(self._nn_j[i, j])
            source = 'copernicus_nearest_ocean'

        uo = float(self._v['uo'][k, i, j])
        vo = float(self._v['vo'][k, i, j])
        SST = float(self._v['thetao'][k, i, j])
        Hs_raw = float(self._v['swh'][k, i, j])
        mwd_raw = float(self._v['mwd'][k, i, j])
        Hs_ww_raw = float(self._v['wind_wave_height'][k, i, j])
        if not all(np.isfinite([uo, vo, SST, Hs_raw, mwd_raw, Hs_ww_raw])):
            raise RuntimeError(f'Weather data has a gap at {pd.Timestamp(timestamp)}, '
                               f'lat {self._lat[i]:.3f}, lon {self._lon[j]:.3f} '
                               '(NaN in an otherwise valid cell) -- no fallback is applied.')

        Hs = float(np.clip(Hs_raw, 0.1, 15.0))
        Tp = float('nan')                                        # VTPK not downloaded; unused by the physics
        mwd = mwd_raw % 360
        Hs_ww = float(np.clip(Hs_ww_raw, 0.0, 12.0))
        wind_ms = wind_speed_from_wind_wave_height(Hs_ww)

        curr_speed_kn = float(np.hypot(uo, vo)) * 1.9438
        curr_dir_deg = (np.degrees(np.arctan2(uo, vo))) % 360
        curr_along_kn = curr_speed_kn * np.cos(np.radians(curr_dir_deg - heading_deg))

        wave_from_deg = mwd
        mu = abs(((wave_from_deg - heading_deg + 180) % 360) - 180)

        Bf = next((b for b, t in enumerate(BEAUFORT_THRESHOLDS) if wind_ms < t), 12)

        return {
            'Hs_m': Hs, 'Tp_s': Tp, 'wind_speed_ms': wind_ms, 'beaufort_number': Bf,
            'wave_encounter_angle_deg': float(mu), 'current_along_track_kn': float(curr_along_kn),
            'SST_C': float(np.clip(SST, -2.0, 36.0)), 'weather_source': source,
        }


## **HOURLY VOYAGE SIMULATION**

In [ ]:
AUX_LOAD_FRAC_MCR = 0.045   # in-port generator load, assumption ( to power lighting, refrigerated containers, pumps, etc.)
AUX_SFOC_G_KWH    = 215.0   # typical diesel-generator SFOC, assumption
PORT_HS_M         = 0.3     # port sea state assumption
HOURS_PER_YEAR    = 8760


class RouteWander:   # to include naturalness of ship route
    def __init__(self, tau_hr=6.0, rng=None):
        self.state = 0.0
        self.tau_hr = tau_hr
        self.rng = rng if rng is not None else np.random.default_rng(2025) # random number generator = rng

    def step(self, dt_hr=1.0):
        phi = np.exp(-dt_hr / self.tau_hr) #decay factor. As time passes (dt_hr), the influence of the previous state fades away exponentially.
        self.state = phi * self.state + np.sqrt(max(1 - phi**2, 0.0)) * self.rng.normal() # blends the old state with a fresh Gaussian random draw (self.rng.normal()), scaled by sqrt(1 - phi^2). This specific math ensures that even as time moves forward, the variance of the noise doesn't expand to infinity).
        self.state = float(np.clip(self.state, -2.5, 2.5)) #ship can't drift more than $\pm 2.5$ units away from the route centreline
        return self.state


def jittered_state_at(interp, dist_nm, Hs, wind_ms, wander):      # Wraps RouteInterpolator. state_at with a small weather-scaled lateral wander.
    lat, lon, hdg = interp.state_at(dist_nm)
    max_offset_nm = float(np.clip(0.12 * Hs + 0.01 * wind_ms, 0.0, 2.5))
    offset_nm = (wander.step() / 2.5) * max_offset_nm
    perp_deg = (hdg + 90.0) % 360.0
    dlat = (offset_nm / 60.0) * np.cos(np.radians(perp_deg))
    dlon = (offset_nm / 60.0) * np.sin(np.radians(perp_deg)) / max(np.cos(np.radians(lat)), 1e-6)
    return lat + dlat, lon + dlon, hdg


def hull_fouling_factor(t_days_since_drydock, hull_condition='well_maintained'):     # multiplier from hull/propeller fouling since the last drydock
    T_cycle_days = 5 * 365.0
    P_max = {'well_maintained': 0.13, 'poorly_maintained': 0.25}.get(hull_condition, 0.13)
    frac = float(np.clip(t_days_since_drydock / T_cycle_days, 0.0, 1.0))
    return 1.0 + P_max * frac ** 1.3


def _sst_at_port(lat, lon, current_time, weather_source):
    return float(weather_source.at(0, 0.0, lat=lat, lon=lon, timestamp=current_time)['SST_C'])


In [ ]:
def _sail_one_hour(ship, interp, wander, weather_source, dist_sailed_nm, leg_type, speed_kn,
                    cargo_full_t, fuel_cum_leg, bog_cum_kg, t_days_since_drydock,
                    hull_condition, current_time):        # Moves one simulated hour along a laden or ballast leg.
    base_lat, base_lon, hdg0 = interp.state_at(dist_sailed_nm)
    wx = weather_source.at(dist_sailed_nm, hdg0, lat=base_lat, lon=base_lon, timestamp=current_time)
    lat, lon, hdg = jittered_state_at(interp, dist_sailed_nm, wx['Hs_m'], wx['wind_speed_ms'], wander)

    Hs, Tp, wind = wx['Hs_m'], wx['Tp_s'], wx['wind_speed_ms']
    mu, SST, curr = wx['wave_encounter_angle_deg'], wx['SST_C'], wx['current_along_track_kn']

    if leg_type == 'laden':
        draft_now = ship['T_laden'] - 0.06 * (fuel_cum_leg / max(cargo_full_t, 1))
    else:
        draft_now = ship['T_ballast']

    V_stw, dV_pct = kwon_speed_loss(speed_kn, Hs, Tp, wind, mu, ship)
    V_sog = float(np.clip(V_stw + curr, speed_kn * 0.35, speed_kn * 1.08))

    P_kW_cw  = admiralty_power(speed_kn, ship, leg_type, draft_now, SST)
    R_add_kN = faltinsen_wave_resistance_kN(Hs, speed_kn, ship, SST)
    P_kW_raw = P_kW_cw + R_add_kN * speed_kn * 0.5144
    foul_mult = hull_fouling_factor(t_days_since_drydock, hull_condition)
    P_kW_total = float(np.clip(P_kW_raw * foul_mult, 50, ship['MCR_kW'] * 1.05))

    eta_total, eta_D, eta_O, eta_H = wageningen_eta(speed_kn, ship, P_kW_total)
    P_shaft_kW = P_kW_total / eta_total
    SFOC, eng_load = engine_sfoc(P_shaft_kW, ship)
    fuel_hr = P_shaft_kW * SFOC * 1.0 / 1e6
    bog_hr_kg = boil_off_rate(ship, SST, Hs, dt_hr=1.0) if leg_type == 'laden' else 0.0

    new_fuel_cum_leg = fuel_cum_leg + fuel_hr
    new_bog_cum_kg = bog_cum_kg + bog_hr_kg
    cargo_remaining = (float(np.clip(cargo_full_t - new_bog_cum_kg / 1000.0, 0, cargo_full_t))
                        if leg_type == 'laden' else 0.0)

    row = {
        'lat': round(lat, 4), 'lon': round(lon, 4), 'ship_heading_deg': round(hdg, 1),
        'dist_sailed_nm': round(dist_sailed_nm, 2),
        'Hs_m': Hs, 'Tp_s': Tp, 'wind_speed_ms': wind, 'beaufort_number': wx['beaufort_number'],
        'wave_encounter_angle_deg': mu, 'current_along_track_kn': curr, 'SST_C': SST,
        'speed_loss_pct': round(dV_pct, 3), 'speed_through_water_kn': round(V_stw, 3),
        'speed_over_ground_kn': round(V_sog, 3),
        'power_calm_kW': round(P_kW_cw, 1), 'wave_resistance_kN': round(R_add_kN, 2),
        'power_total_kW': round(P_kW_total, 1),
        'eta_hull': round(eta_H, 4), 'eta_open_water': round(eta_O, 4), 'eta_propulsive': round(eta_D, 4),
        'SFOC_g_kWh': round(SFOC, 1), 'engine_load_pct': round(eng_load * 100, 2),
        'fuel_this_hour_t': round(fuel_hr, 6), 'bog_this_hour_kg': round(bog_hr_kg, 4),
        'cargo_remaining_t': round(cargo_remaining, 2), 'fuel_cum_leg_t': round(new_fuel_cum_leg, 4),
        'hull_fouling_factor': round(foul_mult, 4),
        'weather_source': wx.get('weather_source', 'copernicus_nearest'),
    }
    new_dist = float(np.clip(dist_sailed_nm + V_sog, 0, interp.total_dist_nm + 1))
    return row, new_dist, new_fuel_cum_leg, new_bog_cum_kg


def _port_hour_row(lat, lon, SST, ship, cargo_aboard_t, t_days_since_drydock, hull_condition):
    """One hour at berth/anchorage: small auxiliary-generator fuel burn;
    BOG keeps accruing whenever cargo is aboard (loaded-waiting + unloading),
    zero otherwise."""
    P_aux_kW = ship['MCR_kW'] * AUX_LOAD_FRAC_MCR
    fuel_hr = P_aux_kW * AUX_SFOC_G_KWH / 1e6
    bog_hr_kg = boil_off_rate(ship, SST, PORT_HS_M, dt_hr=1.0) if cargo_aboard_t > 0 else 0.0
    foul_mult = hull_fouling_factor(t_days_since_drydock, hull_condition)
    return {
        'lat': round(lat, 4), 'lon': round(lon, 4), 'ship_heading_deg': 0.0, 'dist_sailed_nm': 0.0,
        'Hs_m': PORT_HS_M, 'Tp_s': 0.0, 'wind_speed_ms': 0.0, 'beaufort_number': 0,
        'wave_encounter_angle_deg': 0.0, 'current_along_track_kn': 0.0, 'SST_C': round(SST, 1),
        'speed_loss_pct': 0.0, 'speed_through_water_kn': 0.0, 'speed_over_ground_kn': 0.0,
        'power_calm_kW': round(P_aux_kW, 1), 'wave_resistance_kN': 0.0, 'power_total_kW': round(P_aux_kW, 1),
        'eta_hull': 0.0, 'eta_open_water': 0.0, 'eta_propulsive': 0.0,
        'SFOC_g_kWh': AUX_SFOC_G_KWH, 'engine_load_pct': round(AUX_LOAD_FRAC_MCR * 100, 2),
        'fuel_this_hour_t': round(fuel_hr, 6), 'bog_this_hour_kg': round(bog_hr_kg, 4),
        'cargo_remaining_t': round(max(cargo_aboard_t - bog_hr_kg / 1000.0, 0.0), 2),
        'fuel_cum_leg_t': 0.0, 'hull_fouling_factor': round(foul_mult, 4),
        'weather_source': 'port_assumption',
    }


In [ ]:
def simulate_ship_route_year(ship_key, route_key, waypoints, speed_kn, weather_source,
                              hull_condition='well_maintained', rng=None, verbose=True):
    ship = dict(SHIPS[ship_key])
    rng = rng or np.random.default_rng(abs(hash((ship_key, route_key))) % (2**31))
    cargo_full_t = ship['cargo_capacity_t'] * 0.95

    interp_fwd = RouteInterpolator(waypoints)
    interp_rev = RouteInterpolator(list(reversed(waypoints)))
    origin_ll, dest_ll = waypoints[0], waypoints[-1]

    rows = []
    voyage_number = 0
    hour_of_year = 0
    t_days_since_drydock = 0.0
    current_time = pd.Timestamp(f'2025-01-01 00:00:00')

    def _emit(row, leg, position, season):
        nonlocal hour_of_year, current_time, t_days_since_drydock
        row.update({
            'timestamp': current_time, 'ship_size_class': ship_key, 'route_id': route_key,
            'season': season, 'voyage_number': voyage_number,
            'hour_of_voyage': hour_of_year - voyage_start_hour, 'leg': leg, 'position': position,
        })
        rows.append(row)
        hour_of_year += 1
        current_time += pd.Timedelta(hours=1)
        t_days_since_drydock += 1.0 / 24.0

    while hour_of_year < HOURS_PER_YEAR:
        voyage_number += 1
        voyage_start_hour = hour_of_year
        season = MONTH_TO_SEASON[current_time.month]
        wander_laden = RouteWander(rng=rng)
        wander_ballast = RouteWander(rng=rng)

        # LOADING OF SHIP
        port_load_hr = (cargo_full_t / ship['loading_rate_t_hr']
                         + rng.uniform(3, 7) + rng.uniform(0.5, 2.0))
        n_load = int(np.ceil(port_load_hr))
        for h in range(n_load):
            if hour_of_year >= HOURS_PER_YEAR:
                break
            SST = _sst_at_port(*origin_ll, current_time, weather_source)
            cargo_now = cargo_full_t * (h + 1) / max(n_load, 1)
            row = _port_hour_row(origin_ll[0], origin_ll[1], SST, ship, cargo_now,
                                  t_days_since_drydock, hull_condition)
            _emit(row, 'port', 'loading', season)
        if hour_of_year >= HOURS_PER_YEAR:
            break

        # WAITING
        weather_wait_hr = float(rng.exponential(SEASONAL_PORT_WAIT_HR[season]))
        hurricane_delay_hr = 0.0
        if season in ('Summer', 'Autumn') and rng.random() > HURRICANE_OPS_FACTOR[season]:
            hurricane_delay_hr = float(rng.exponential(36.0))
        n_wait = int(np.ceil(weather_wait_hr + hurricane_delay_hr))
        for h in range(n_wait):
            if hour_of_year >= HOURS_PER_YEAR:
                break
            SST = _sst_at_port(*origin_ll, current_time, weather_source)
            row = _port_hour_row(origin_ll[0], origin_ll[1], SST, ship, cargo_full_t,
                                  t_days_since_drydock, hull_condition)
            _emit(row, 'port', 'waiting_port', season)
        if hour_of_year >= HOURS_PER_YEAR:
            break

        # LADEN TRANSPORT
        dist_sailed, fuel_cum_l, bog_cum_kg = 0.0, 0.0, 0.0
        while dist_sailed < interp_fwd.total_dist_nm and hour_of_year < HOURS_PER_YEAR:
            row, dist_sailed, fuel_cum_l, bog_cum_kg = _sail_one_hour(
                ship, interp_fwd, wander_laden, weather_source, dist_sailed, 'laden',
                speed_kn, cargo_full_t, fuel_cum_l, bog_cum_kg, t_days_since_drydock,
                hull_condition, current_time)
            _emit(row, 'laden', 'sailing_laden', season)
            if dist_sailed >= interp_fwd.total_dist_nm:
                break
        bog_cum_kg_laden = bog_cum_kg
        if hour_of_year >= HOURS_PER_YEAR:
            break

        # UNLOADING OF SHIP
        net_co2_delivered_t = float(np.clip(cargo_full_t - bog_cum_kg_laden / 1000.0, 0, cargo_full_t))
        port_unload_hr = (cargo_full_t / ship['unloading_rate_t_hr']
                           + rng.uniform(3, 7) + rng.uniform(0.5, 2.0))
        n_unload = int(np.ceil(port_unload_hr))
        for h in range(n_unload):
            if hour_of_year >= HOURS_PER_YEAR:
                break
            SST = _sst_at_port(*dest_ll, current_time, weather_source)
            cargo_now = max(net_co2_delivered_t * (1 - (h + 1) / max(n_unload, 1)), 0.0)
            row = _port_hour_row(dest_ll[0], dest_ll[1], SST, ship, cargo_now,
                                  t_days_since_drydock, hull_condition)
            _emit(row, 'port', 'unloading', season)
        if hour_of_year >= HOURS_PER_YEAR:
            break

        # BALLAST TRANSPORT
        dist_sailed, fuel_cum_b = 0.0, 0.0
        while dist_sailed < interp_rev.total_dist_nm and hour_of_year < HOURS_PER_YEAR:
            row, dist_sailed, fuel_cum_b, _ = _sail_one_hour(
                ship, interp_rev, wander_ballast, weather_source, dist_sailed, 'ballast',
                speed_kn, cargo_full_t, fuel_cum_b, 0.0, t_days_since_drydock,
                hull_condition, current_time)
            _emit(row, 'ballast', 'sailing_ballast', season)
            if dist_sailed >= interp_rev.total_dist_nm:
                break

        if verbose and (voyage_number % 10 == 0):
            print(f"    ...{ship_key}/{route_key}: voyage {voyage_number:4d} done "
                  f"({hour_of_year}/{HOURS_PER_YEAR} hours)")

    df_hourly = pd.DataFrame(rows[:HOURS_PER_YEAR])
    df_voyages = _build_voyage_summary(df_hourly, ship_key, {route_key: interp_fwd.total_dist_nm})
    return df_hourly, df_voyages

## **VOYAGE SUMMARY**

In [ ]:
def _build_voyage_summary(df_hourly, ship_key, route_total_dist=None):
    ship = SHIPS[ship_key]
    cargo_loaded_t = ship['cargo_capacity_t'] * 0.95
    route_total_dist = route_total_dist or {}
    out_rows = []
    for voyage_number, g in df_hourly.groupby('voyage_number', sort=True):
        laden     = g[g['leg'] == 'laden']
        ballast   = g[g['leg'] == 'ballast']
        port      = g[g['leg'] == 'port']
        unloading = g[g['position'] == 'unloading']
        loading   = g[g['position'] == 'loading']
        route_id  = g['route_id'].iloc[0]

        laden_hours, ballast_hours = len(laden), len(ballast)
        fuel_laden_t   = float(laden['fuel_this_hour_t'].sum())
        fuel_ballast_t = float(ballast['fuel_this_hour_t'].sum())
        fuel_port_t    = float(port['fuel_this_hour_t'].sum())
        total_fuel_t   = fuel_laden_t + fuel_ballast_t + fuel_port_t

        bog_laden_kg = float(laden['bog_this_hour_kg'].sum())
        bog_total_kg = float(g['bog_this_hour_kg'].sum())

        target_dist = route_total_dist.get(route_id)
        laden_reached_dest = bool(laden_hours and (
            len(unloading) > 0 or len(ballast) > 0 or
            (target_dist is not None and float(laden['dist_sailed_nm'].max()) >= target_dist - 0.5)))
        net_co2_delivered_t = (float(np.clip(cargo_loaded_t - bog_laden_kg / 1000.0, 0, cargo_loaded_t))
                                if laden_reached_dest else 0.0)

        out_rows.append({
            'ship_size_class': ship_key, 'voyage_number': voyage_number,
            'route_id': route_id, 'season': g['season'].iloc[0],
            'route_dist_nm': round(float(laden['dist_sailed_nm'].max()), 1) if laden_hours else np.nan,
            'laden_hours': laden_hours, 'ballast_hours': ballast_hours,
            'total_sailing_hr': laden_hours + ballast_hours, 'total_port_hr': len(port),
            'total_voyage_hr': len(g), 'laden_leg_delivered': laden_reached_dest,
            'is_complete_voyage': bool(laden_hours and ballast_hours and len(loading) and len(unloading)),
            'mean_sog_laden_kn':   round(float(laden['speed_over_ground_kn'].mean()), 3) if laden_hours else 0.0,
            'mean_speed_loss_pct': round(float(laden['speed_loss_pct'].mean()), 3) if laden_hours else 0.0,
            'max_speed_loss_pct':  round(float(laden['speed_loss_pct'].max()), 3) if laden_hours else 0.0,
            'mean_Hs_laden_m':     round(float(laden['Hs_m'].mean()), 3) if laden_hours else 0.0,
            'max_Hs_laden_m':      round(float(laden['Hs_m'].max()), 3) if laden_hours else 0.0,
            'mean_wind_ms':        round(float(laden['wind_speed_ms'].mean()), 2) if laden_hours else 0.0,
            'mean_SST_C':          round(float(laden['SST_C'].mean()), 1) if laden_hours else 0.0,
            'max_SST_C':           round(float(laden['SST_C'].max()), 1) if laden_hours else 0.0,
            'hours_Hs_gt_15m':     int((laden['Hs_m'] > 1.5).sum()),
            'hours_Hs_gt_2m':      int((laden['Hs_m'] > 2.0).sum()),
            'mean_power_laden_kW': round(float(laden['power_total_kW'].mean()), 1) if laden_hours else 0.0,
            'mean_engine_load_pct':round(float(laden['engine_load_pct'].mean()), 2) if laden_hours else 0.0,
            'mean_eta_propulsive': round(float(laden['eta_propulsive'].mean()), 4) if laden_hours else 0.0,
            'fuel_laden_t': round(fuel_laden_t, 3), 'fuel_ballast_t': round(fuel_ballast_t, 3),
            'fuel_port_t': round(fuel_port_t, 3), 'total_fuel_t': round(total_fuel_t, 3),
            'fuel_t_per_day_sailing': round((fuel_laden_t + fuel_ballast_t)
                                             / max(laden_hours + ballast_hours, 1) * 24, 2),
            'mean_SFOC_g_kWh': round(float(laden['SFOC_g_kWh'].mean()), 1) if laden_hours else 0.0,
            'cargo_loaded_t': round(cargo_loaded_t, 1),
            'bog_total_kg': round(bog_total_kg, 2), 'bog_total_t': round(bog_total_kg / 1000.0, 4),
            'BOR_pct_day': round(bog_laden_kg / 1000.0 / cargo_loaded_t / (laden_hours / 24) * 100, 4)
                            if laden_hours > 0 else 0.0,
            'net_co2_delivered_t': round(net_co2_delivered_t, 2),
            'ship_co2_emitted_t': round(total_fuel_t * ECON['co2_per_t_vlsfo'], 3),
            'voyage_efficiency_pct': round(net_co2_delivered_t / cargo_loaded_t * 100, 3) if cargo_loaded_t else 0.0,
            'transport_efficiency_t_t': round(net_co2_delivered_t / total_fuel_t, 2) if total_fuel_t > 0 else 0.0,
        })
    return pd.DataFrame(out_rows)

In [ ]:
VOYAGE_BROADCAST_COLS = [
    'voyage_number', 'mean_sog_laden_kn', 'mean_speed_loss_pct', 'max_speed_loss_pct',
    'mean_Hs_laden_m', 'max_Hs_laden_m', 'mean_wind_ms', 'mean_SST_C', 'max_SST_C',
    'hours_Hs_gt_15m', 'hours_Hs_gt_2m', 'mean_power_laden_kW', 'mean_engine_load_pct',
    'mean_eta_propulsive', 'fuel_laden_t', 'fuel_ballast_t', 'total_fuel_t',
    'fuel_t_per_day_sailing', 'mean_SFOC_g_kWh', 'cargo_loaded_t', 'bog_total_kg',
    'bog_total_t', 'BOR_pct_day', 'net_co2_delivered_t', 'ship_co2_emitted_t',
    'voyage_efficiency_pct', 'transport_efficiency_t_t',
]


def add_voyage_broadcast_columns(df_hourly, df_voyages):
    return df_hourly.merge(df_voyages[VOYAGE_BROADCAST_COLS], on='voyage_number', how='left')

## **TECHNO-ECONOMIC ANALYSIS**

In [ ]:
ANNUAL_BROADCAST_COLS = [
    'annual_co2_per_ship_t', 'vessel_utilisation', 'ship_opex_annual_musd',
    'fuel_cost_annual_musd', 'carbon_cost_annual_musd', 'port_cost_annual_musd',
    'storage_opex_annual_musd', 'total_opex_annual_musd', 'liq_opex_annual_musd',
    'total_cost_annual_musd', 'transport_cost_usd_per_t', 'liq_cost_usd_per_t',
    'full_chain_cost_usd_per_t', 'co2_emitted_annual_kt', 'net_transport_eff_pct',
    'specific_emission_kg_t_km',
]


In [ ]:
def compute_annual_tea(df_voyages, ship_key):  # summary for one ship+route's simulated year. Port calls are approximated as 2 per voyage (one load + one unload)
    ship = SHIPS[ship_key]
    n_voyages = len(df_voyages)
    total_hours_sim = float(df_voyages['total_voyage_hr'].sum()) or float(HOURS_PER_YEAR)

    annual_co2_delivered_t = float(df_voyages['net_co2_delivered_t'].sum())
    annual_fuel_t = float(df_voyages['total_fuel_t'].sum())
    annual_bog_t = float(df_voyages['bog_total_t'].sum())
    co2_emitted_annual_t = float(df_voyages['ship_co2_emitted_t'].sum())
    n_port_calls = 2 * n_voyages

    ship_opex_annual_musd = ECON['ship_opex_usd_per_day'][ship_key] * 365 / 1e6
    fuel_cost_annual_musd = annual_fuel_t * ECON['fuel_price_usd_per_t'] / 1e6
    carbon_cost_annual_musd = co2_emitted_annual_t * ECON['carbon_price_usd_per_t_co2'] / 1e6
    port_cost_annual_musd = n_port_calls * ECON['port_cost_per_call_usd'][ship_key] / 1e6

    avg_daily_throughput_t = annual_co2_delivered_t / 365.0
    buffer_capacity_t = avg_daily_throughput_t * ECON['buffer_storage_days']
    storage_opex_annual_musd = buffer_capacity_t * ECON['buffer_storage_opex_usd_per_t_cap_yr'] / 1e6

    total_opex_annual_musd = (ship_opex_annual_musd + fuel_cost_annual_musd + carbon_cost_annual_musd
                               + port_cost_annual_musd + storage_opex_annual_musd)
    liq_opex_annual_musd = annual_co2_delivered_t * ECON['liquefaction_opex_usd_per_t'] / 1e6
    total_cost_annual_musd = total_opex_annual_musd + liq_opex_annual_musd

    have_cargo = annual_co2_delivered_t > 0
    transport_cost_usd_per_t = (total_opex_annual_musd * 1e6 / annual_co2_delivered_t) if have_cargo else np.nan
    liq_cost_usd_per_t = ECON['liquefaction_opex_usd_per_t']
    full_chain_cost_usd_per_t = (transport_cost_usd_per_t + liq_cost_usd_per_t) if have_cargo else np.nan

    co2_emitted_annual_kt = co2_emitted_annual_t / 1000.0
    gross_handled_t = annual_co2_delivered_t + co2_emitted_annual_t
    net_transport_eff_pct = (annual_co2_delivered_t / gross_handled_t * 100) if gross_handled_t > 0 else 0.0

    total_km_sailed = float(df_voyages['route_dist_nm'].fillna(0).sum()) * 2 * 1.852  # round trip, nm->km
    specific_emission_kg_t_km = ((co2_emitted_annual_t * 1000) / (annual_co2_delivered_t * total_km_sailed)
                                  if (have_cargo and total_km_sailed > 0) else np.nan)

    vessel_utilisation = float(df_voyages['total_sailing_hr'].sum()) / total_hours_sim * 100

    return {
        'annual_co2_per_ship_t': round(annual_co2_delivered_t, 1),
        'vessel_utilisation': round(vessel_utilisation, 2),
        'ship_opex_annual_musd': round(ship_opex_annual_musd, 4),
        'fuel_cost_annual_musd': round(fuel_cost_annual_musd, 4),
        'carbon_cost_annual_musd': round(carbon_cost_annual_musd, 4),
        'port_cost_annual_musd': round(port_cost_annual_musd, 4),
        'storage_opex_annual_musd': round(storage_opex_annual_musd, 4),
        'total_opex_annual_musd': round(total_opex_annual_musd, 4),
        'liq_opex_annual_musd': round(liq_opex_annual_musd, 4),
        'total_cost_annual_musd': round(total_cost_annual_musd, 4),
        'transport_cost_usd_per_t': round(transport_cost_usd_per_t, 3) if have_cargo else np.nan,
        'liq_cost_usd_per_t': round(liq_cost_usd_per_t, 3),
        'full_chain_cost_usd_per_t': round(full_chain_cost_usd_per_t, 3) if have_cargo else np.nan,
        'co2_emitted_annual_kt': round(co2_emitted_annual_kt, 4),
        'net_transport_eff_pct': round(net_transport_eff_pct, 3),
        'specific_emission_kg_t_km': (round(specific_emission_kg_t_km, 6)
                                       if not np.isnan(specific_emission_kg_t_km) else np.nan),
        'n_voyages_completed': n_voyages,
        'annual_fuel_t': round(annual_fuel_t, 2),
        'annual_bog_t': round(annual_bog_t, 3),
    }


def add_annual_broadcast_columns(df_hourly, annual_dict):
    df = df_hourly.copy()
    for col in ANNUAL_BROADCAST_COLS:
        df[col] = annual_dict.get(col, np.nan)
    return df

## **GENERATING THE DATA FILES**

In [ ]:
HOURLY_OUTPUT_COLUMNS = [
    # identity
    'ship_size_class', 'route_id', 'timestamp',
    # hour-level
    'voyage_number', 'hour_of_voyage', 'leg', 'lat', 'lon', 'ship_heading_deg', 'dist_sailed_nm',
    'Hs_m', 'Tp_s', 'wind_speed_ms', 'wave_encounter_angle_deg',
    'current_along_track_kn', 'SST_C', 'speed_loss_pct', 'speed_through_water_kn',
    'speed_over_ground_kn', 'power_calm_kW', 'wave_resistance_kN', 'power_total_kW',
    'eta_hull', 'eta_open_water', 'eta_propulsive', 'SFOC_g_kWh', 'engine_load_pct',
    'fuel_this_hour_t', 'bog_this_hour_kg', 'cargo_remaining_t', 'fuel_cum_leg_t', 'position',
    'hull_fouling_factor',
    # voyage-broadcast
    'mean_sog_laden_kn', 'mean_speed_loss_pct', 'max_speed_loss_pct', 'mean_Hs_laden_m',
    'max_Hs_laden_m', 'mean_wind_ms', 'mean_SST_C', 'max_SST_C', 'hours_Hs_gt_15m',
    'hours_Hs_gt_2m', 'mean_power_laden_kW', 'mean_engine_load_pct', 'mean_eta_propulsive',
    'fuel_laden_t', 'fuel_ballast_t', 'total_fuel_t', 'fuel_t_per_day_sailing',
    'mean_SFOC_g_kWh', 'cargo_loaded_t', 'bog_total_kg', 'bog_total_t', 'BOR_pct_day',
    'net_co2_delivered_t', 'ship_co2_emitted_t', 'voyage_efficiency_pct', 'transport_efficiency_t_t',
    # annual-broadcast
    'annual_co2_per_ship_t', 'vessel_utilisation', 'ship_opex_annual_musd',
    'fuel_cost_annual_musd', 'carbon_cost_annual_musd', 'port_cost_annual_musd',
    'storage_opex_annual_musd', 'total_opex_annual_musd', 'liq_opex_annual_musd',
    'total_cost_annual_musd', 'transport_cost_usd_per_t', 'liq_cost_usd_per_t',
    'full_chain_cost_usd_per_t', 'co2_emitted_annual_kt', 'net_transport_eff_pct',
    'specific_emission_kg_t_km',
]

In [ ]:
def finalize_hourly_file(df_hourly, df_voyages, annual_tea):
    df = add_voyage_broadcast_columns(df_hourly, df_voyages)
    df = add_annual_broadcast_columns(df, annual_tea)
    cols = [c for c in HOURLY_OUTPUT_COLUMNS if c in df.columns]
    missing = [c for c in HOURLY_OUTPUT_COLUMNS if c not in df.columns]
    if missing:
        print(f'  [WARN] finalize_hourly_file: columns missing from output (skipped): {missing}')
    return df[cols]


def run_all_ship_route_pairs( weather_source, hull_condition='well_maintained', verbose=True):
    files, all_voyages, all_tea, finalized = {}, {}, {}, {}
    for ship_key in SHIPS:
        for route_key, route in ROUTES.items():
            t0 = time.time()
            print(f"  Simulating {ship_key} on {route_key} ({route['name']}) .")
            rng = np.random.default_rng(abs(hash((ship_key, route_key))) % (2**31))
            df_hourly, df_voyages = simulate_ship_route_year(
                ship_key, route_key, route['waypoints'], ship_speed,
                 weather_source, hull_condition=hull_condition, rng=rng, verbose=verbose)
            annual_tea = compute_annual_tea(df_voyages, ship_key)
            df_final = finalize_hourly_file(df_hourly, df_voyages, annual_tea)

            fname = f'lco2_hourly_{ship_key}_{route_key}.csv'
            fpath = os.path.join(OUTPUT_DIR, fname)
            df_final.to_csv(fpath, index=False)

            files[(ship_key, route_key)] = fpath
            all_voyages[(ship_key, route_key)] = df_voyages
            all_tea[(ship_key, route_key)] = annual_tea
            finalized[(ship_key, route_key)] = df_final
            print(f"    -> {fname}  ({len(df_final):,} rows, {df_voyages.shape[0]} voyages, "
                  f"{time.time() - t0:.1f}s)")

    combined_df = pd.concat(finalized.values(), ignore_index=True)
    combined_path = os.path.join(OUTPUT_DIR, f'lco2_all_ships_routes.csv')
    combined_df.to_csv(combined_path, index=False)
    print(f"  -> {os.path.basename(combined_path)}  ({len(combined_df):,} rows total, "
          f"{len(files)} ship/route files combined)")

    return files, all_voyages, all_tea, combined_df, combined_path

## **COMPARISION OF ROUTES AND SHIPS**

In [ ]:
def build_comparison_table(all_voyages, all_tea):
    rows = []
    for (ship_key, route_key), g in all_voyages.items():
        delivered = g[g['laden_leg_delivered']]
        eff = g.loc[g['transport_efficiency_t_t'] > 0, 'transport_efficiency_t_t']
        tea = all_tea[(ship_key, route_key)]
        rows.append({
            'ship_size_class': ship_key, 'route_id': route_key,
            'route_name': ROUTES[route_key]['name'], 'n_voyages': len(g),
            'mean_route_dist_nm': round(float(g['route_dist_nm'].mean()), 1),
            'mean_fuel_t_per_day_sailing': round(float(g['fuel_t_per_day_sailing'].mean()), 2),
            'mean_speed_loss_pct': round(float(g['mean_speed_loss_pct'].mean()), 2),
            'mean_Hs_laden_m': round(float(g['mean_Hs_laden_m'].mean()), 2),
            'mean_BOR_pct_day': round(float(g['BOR_pct_day'].mean()), 4),
            'total_net_co2_delivered_t': round(float(delivered['net_co2_delivered_t'].sum()), 1),
            'mean_transport_efficiency_t_t': round(float(eff.mean()), 2) if len(eff) else np.nan,
            'total_fuel_t': round(float(g['total_fuel_t'].sum()), 1),
            'vessel_utilisation_pct': tea['vessel_utilisation'],
            'transport_cost_usd_per_t': tea['transport_cost_usd_per_t'],
            'full_chain_cost_usd_per_t': tea['full_chain_cost_usd_per_t'],
            'co2_emitted_annual_kt': tea['co2_emitted_annual_kt'],
            'net_transport_eff_pct': tea['net_transport_eff_pct'],
        })
    return pd.DataFrame(rows).sort_values(['route_id', 'ship_size_class']).reset_index(drop=True)

## **MAIN FUNCTION**

In [ ]:
def run_full_pipeline(hull_condition='well_maintained',
                       weather_dir=None, fill_land_with_nearest_ocean=True, verbose=True):
    t_start = time.time()
    print(f"LCO2 GULF TRANSPORT SIMULATION ")
    print(f"  Ships:  {', '.join(SHIPS)}")
    print(f"  Routes: {', '.join(f'{k} ({v['name']})' for k, v in ROUTES.items())}")
    print(f"  Output: {OUTPUT_DIR}")

    # Weather
    print("Weather Part")
    weather_source = CopernicusWeather(WEATHER_DIR,
                                       fill_land_with_nearest_ocean=fill_land_with_nearest_ocean)

    # Simulate all 9 pairs and write files
    print(f"\n Simulating {len(SHIPS)} ships and {len(ROUTES)} routes ")
    files, all_voyages, all_tea, combined_df, combined_path = run_all_ship_route_pairs( weather_source, hull_condition=hull_condition, verbose=verbose)

    # Compare results
    print("\n Cross-ship and cross-route comparison")
    comparison_df = build_comparison_table(all_voyages, all_tea)
    comparison_path = os.path.join(OUTPUT_DIR, f'lco2_ship_route_comparison.csv')
    comparison_df.to_csv(comparison_path, index=False)
    print(f" to {os.path.basename(comparison_path)}")

    print("DONE")

    return {
        'files': files, 'combined_path': combined_path, 'combined_df': combined_df,
        'comparison_path': comparison_path, 'comparison_df': comparison_df,
        'all_voyages': all_voyages, 'all_tea': all_tea,
    }


if __name__ == '__main__':
    run_full_pipeline()


LCO2 GULF TRANSPORT SIMULATION 
  Ships:  S10, S30, S50
  Routes: GOM-HSC_BB (GOM-HSC_BB), GOM-PAB_BB (GOM-PAB_BB), GOM-PLQ_BB (GOM-PLQ_BB)
  Output: /content/drive/MyDrive/LCO2 Transport Project Final/simulation_files
Weather Part
  [Weather] cropping to route box lon -95.12..-88.91, lat 28.36..30.64 before loading
    loaded gulf_merged_2025_01.nc  (744 steps)
    loaded gulf_merged_2025_02.nc  (672 steps)
    loaded gulf_merged_2025_03.nc  (744 steps)
    loaded gulf_merged_2025_04.nc  (720 steps)
    loaded gulf_merged_2025_05.nc  (744 steps)
    loaded gulf_merged_2025_06.nc  (720 steps)
    loaded gulf_merged_2025_07.nc  (744 steps)
    loaded gulf_merged_2025_08.nc  (744 steps)
    loaded gulf_merged_2025_09.nc  (720 steps)
    loaded gulf_merged_2025_10.nc  (744 steps)
    loaded gulf_merged_2025_11.nc  (720 steps)
    loaded gulf_merged_2025_12.nc  (744 steps)
  [Weather] ready: 8760 time steps (2025-01-01 00:00:00 .. 2025-12-31 23:00:00), grid 27 lat x 75 lon, 971 no-data (la